![OneHealth DataSpace](https://bigdata.dataspace.cesga.es/static/images/public/imagotipo.png)

> **⚠️ ADVERTENCIA**: actualmente la plataforma está en fase de pueba. Agradecemos que nos hagáis llegar cualquier comentario a onehealth@cesga.es

En este notebook veremos cómo analizar la profundidad de captura del bacalao utilizando los datos de capturas de DIVERSIMAR de forma totalmente interactiva.

In [ ]:
from bokeh.models import FixedTicker, NumeralTickFormatter

opciones = ["Todas las especies"] + listado_especies

especie_inicial = "Pulpo blanco"

p = figure(
    title=f"Profundidad de captura {especie_inicial}",
    x_axis_label="Profundidad (m)",
    y_axis_label="Capturas",
    height=650,
    width=950
)

renderers = {}

for especie in listado_especies:
    source = ColumnDataSource(data=datos[especie])

    renderers[especie] = p.vbar(
        x="x",
        top="top",
        bottom=0,
        width=50,
        source=source,
        line_color="black",
        fill_color="color",
        alpha=0.6,
        visible=(especie == especie_inicial),
        legend_label=especie
    )

p.legend.location = "top_right"
p.legend.click_policy = "hide"

# Eje X en números normales: 0, 100, 200...
p.xaxis.ticker = FixedTicker(ticks=list(range(0, 1551, 100)))
p.xaxis.formatter = NumeralTickFormatter(format="0")
p.xaxis.major_label_text_font_size = "11pt"
p.yaxis.formatter = NumeralTickFormatter(format="0,0")
p.yaxis.major_label_text_font_size = "11pt"

p.min_border_bottom = 70
p.min_border_left = 80

select = Select(
    title="Especie",
    value=especie_inicial,
    options=opciones
)

callback = CustomJS(
    args=dict(renderers=renderers, p=p),
    code="""
    const especie = cb_obj.value;

    if (especie === "Todas las especies") {
        for (const nombre in renderers) {
            renderers[nombre].visible = true;
        }
        p.title.text = "Profundidad de captura de todas as especies";
    } else {
        for (const nombre in renderers) {
            renderers[nombre].visible = (nombre === especie);
        }
        p.title.text = "Profundidad de captura " + especie;
    }
    """
)

select.js_on_change("value", callback)

show(column(select, p))